In [2]:
import numpy as np
import random
import torch
from datasets import load_dataset
from transformers import(
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score ,f1_score


In [3]:
seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [4]:
dataset=load_dataset('stanfordnlp/imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
print(dataset)
text=dataset['train']['text']
label=dataset['train']['label']

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [6]:
for i, t in enumerate(enumerate(text)):
    if i >= 10:
        break
    print(t)
for i, t in enumerate(enumerate(label)):
    if i >= 5:
        break
    print(t)

(0, 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between

In [7]:
train_data = dataset["train"]
test_data = dataset["test"].shuffle(seed=seed).select(range(500))
val_data = dataset["test"].shuffle(seed=seed).select(range(500, 700))

In [8]:
model_name='bert-base-uncased'
max_length=256
tokenizer=AutoTokenizer.from_pretrained(model_name)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
def tokenize(batch):
    return tokenizer(batch['text'],truncation=True ,max_length=max_length)

train_tz=train_data.map(tokenize,batched=True)
test_tz=test_data.map(tokenize,batched=True)
val_tz=val_data.map(tokenize,batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [10]:
model=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
def compute_matrics(eval_pred):
    logist,labels=eval_pred
    pred=np.argmax(logist,axis=1)
    return {
        "accuracy":accuracy_score(labels,pred),
        "f1":f1_score(labels,pred)
    }


In [12]:
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)


In [13]:
training_args=TrainingArguments(
    output_dir="./imdb output",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=seed,
    logging_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tz,
    eval_dataset=val_tz,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_matrics
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.257266,0.265883,0.910000,0.892857
2,0.271839,0.364959,0.900000,0.888889


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6250, training_loss=0.2432029579925537, metrics={'train_runtime': 1266.3178, 'train_samples_per_second': 39.485, 'train_steps_per_second': 4.936, 'total_flos': 6573011607231840.0, 'train_loss': 0.2432029579925537, 'epoch': 2.0})

In [15]:
final_result=trainer.evaluate(eval_dataset=test_tz)
print('final results:')
print(final_result)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.271839,0.353072,2,0.920000,0.918699


final results:
{'eval_loss': 0.35307151079177856, 'eval_accuracy': 0.92, 'eval_f1': 0.9186991869918699}


predict sentiment on new reviews

In [24]:
def predict_sentiment(review_text):
    inputs = tokenizer(review_text, return_tensors='pt', truncation=True, padding=True, max_length=max_length)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predicted_label = torch.argmax(probabilities, dim=-1).item()
    sentiment = "Positive" if predicted_label == 1 else "Negative"
    return sentiment, probabilities.cpu().numpy()[0]

In [29]:
predict_sentiment("this movie i dont know but i dont really like it although is good but not my type")

('Positive', array([0.4681219, 0.5318781], dtype=float32))

In [30]:
predict_sentiment("i was okay-some good moments but overall forgettable")


('Negative', array([0.9989617 , 0.00103829], dtype=float32))

saving model

In [ ]:
dir="./model_bert"
trainer.save_model(dir)
tokenizer.save_pretrained(dir)